# Synthetic Artifact Features

The competition data is synthetically generated.

Repeated and discretized numerical values may therefore contain
generator-specific predictive structure that ordinary tree models
do not fully exploit.

This experiment focuses first on Annual_Income_USD.

All target encoding is performed out-of-fold to avoid leakage.

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold
)

from sklearn.metrics import roc_auc_score

from lightgbm import (
    LGBMClassifier,
    early_stopping,
    log_evaluation
)

In [4]:
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

y = train["Will_Buy_EV"].map({
    "No": 0,
    "Yes": 1
})

train.shape, test.shape

((668665, 15), (286571, 14))

In [5]:
print(
    "Rows:",
    len(train)
)

print(
    "Unique incomes:",
    train["Annual_Income_USD"].nunique()
)

print(
    "Average rows per income:",
    len(train)
    / train["Annual_Income_USD"].nunique()
)

Rows: 668665
Unique incomes: 13214
Average rows per income: 50.60276978961707


In [6]:
train[
    "Annual_Income_USD"
].value_counts().describe()

count    13214.000000
mean        50.602770
std        540.861771
min          1.000000
25%          1.000000
50%          8.000000
75%         66.000000
max      61605.000000
Name: count, dtype: float64

In [7]:
train[
    "Annual_Income_USD"
].value_counts().head(20)

Annual_Income_USD
30000.0     61605
86095.0      1082
96749.0      1073
65800.0      1029
92372.0       907
84578.0       905
93957.0       852
68480.0       762
82440.0       739
84768.0       725
80777.0       720
91412.0       712
123547.0      691
94863.0       687
74981.0       680
72441.0       674
91560.0       663
86844.0       659
72454.0       644
94575.0       627
Name: count, dtype: int64

In [8]:
def make_artifact_keys(df):
    keys = pd.DataFrame(
        index=df.index
    )

    income = df[
        "Annual_Income_USD"
    ]

    # Exact synthetic value
    keys["Income_Exact"] = (
        income
        .round(0)
        .astype("Int64")
        .astype(str)
    )

    # $50 bins
    keys["Income_Floor_50"] = (
        np.floor(income / 50) * 50
    ).astype("Int64").astype(str)

    # $250 bins
    keys["Income_Floor_250"] = (
        np.floor(income / 250) * 250
    ).astype("Int64").astype(str)

    # Slightly coarser comparison
    keys["Income_Floor_500"] = (
        np.floor(income / 500) * 500
    ).astype("Int64").astype(str)

    return keys

In [9]:
artifact_keys = make_artifact_keys(
    train
)

artifact_keys.head()

,Income_Exact,Income_Floor_50,Income_Floor_250,Income_Floor_500
0,92887,92850,92750,92500
1,30000,30000,30000,30000
2,94389,94350,94250,94000
3,73580,73550,73500,73500
4,57898,57850,57750,57500


In [10]:
artifact_keys.nunique()

Income_Exact        13214
Income_Floor_50      2026
Income_Floor_250      533
Income_Floor_500      284
dtype: int64

In [11]:
def build_smoothed_mapping(
    key,
    target,
    smoothing=50
):
    temp = pd.DataFrame({
        "key": key
            .astype(str)
            .to_numpy(),

        "target": target
            .to_numpy()
    })

    global_mean = target.mean()

    stats = (
        temp
        .groupby("key")["target"]
        .agg(["sum", "count"])
    )

    stats["encoding"] = (
        stats["sum"]
        + smoothing * global_mean
    ) / (
        stats["count"]
        + smoothing
    )

    return (
        stats["encoding"],
        global_mean
    )

In [12]:
def create_artifact_encodings(
    train_df,
    valid_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(
        drop=True
    )

    valid_df = valid_df.reset_index(
        drop=True
    )

    y_train = y_train.reset_index(
        drop=True
    )

    train_keys = make_artifact_keys(
        train_df
    )

    valid_keys = make_artifact_keys(
        valid_df
    )

    train_encoded = pd.DataFrame(
        index=train_df.index
    )

    valid_encoded = pd.DataFrame(
        index=valid_df.index
    )

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        oof_values = np.zeros(
            len(train_df)
        )

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):

            mapping, global_mean = (
                build_smoothed_mapping(
                    train_keys[col].iloc[
                        inner_train_idx
                    ],
                    y_train.iloc[
                        inner_train_idx
                    ],
                    smoothing=smoothing
                )
            )

            oof_values[
                inner_valid_idx
            ] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_encoded[
            f"TE_{col}"
        ] = oof_values

        full_mapping, full_global_mean = (
            build_smoothed_mapping(
                train_keys[col],
                y_train,
                smoothing=smoothing
            )
        )

        valid_encoded[
            f"TE_{col}"
        ] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

    return (
        train_encoded,
        valid_encoded
    )

In [13]:
def create_frequency_features(
    train_df,
    valid_df
):
    train_df = train_df.reset_index(
        drop=True
    )

    valid_df = valid_df.reset_index(
        drop=True
    )

    train_keys = make_artifact_keys(
        train_df
    )

    valid_keys = make_artifact_keys(
        valid_df
    )

    train_freq = pd.DataFrame(
        index=train_df.index
    )

    valid_freq = pd.DataFrame(
        index=valid_df.index
    )

    for col in train_keys.columns:

        counts = (
            train_keys[col]
            .value_counts()
        )

        train_freq[
            f"FREQ_{col}"
        ] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[
            f"FREQ_{col}"
        ] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_freq,
        valid_freq
    )

In [14]:
train_idx, valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

outer_train = train.iloc[
    train_idx
].copy()

outer_valid = train.iloc[
    valid_idx
].copy()

y_outer_train = y.iloc[
    train_idx
].copy()

y_outer_valid = y.iloc[
    valid_idx
].copy()

In [15]:
train_art_te, valid_art_te = (
    create_artifact_encodings(
        outer_train,
        outer_valid,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42
    )
)

In [16]:
train_art_freq, valid_art_freq = (
    create_frequency_features(
        outer_train,
        outer_valid
    )
)

In [17]:
train_art_te.head()

,TE_Income_Exact,TE_Income_Floor_50,TE_Income_Floor_250,TE_Income_Floor_500
0,0.125568,0.118716,0.100977,0.099945
1,0.237309,0.241288,0.181278,0.245190
2,0.186925,0.221954,0.221018,0.213112
3,0.265811,0.245730,0.230612,0.245034
4,0.193712,0.146562,0.146682,0.192291


In [18]:
train_art_freq.head()

,FREQ_Income_Exact,FREQ_Income_Floor_50,FREQ_Income_Floor_250,FREQ_Income_Floor_500
0,161,600,3537,3708
1,141,305,1426,2043
2,395,1192,1261,2148
3,129,457,1435,4607
4,110,906,922,5538


In [19]:
def add_features(df):
    df = df.copy()

    # Numerical interactions
    df["Income_per_Car"] = (
        df["Annual_Income_USD"]
        / df["Number_of_Cars_Owned"].clip(lower=1)
    )

    df["Total_Charging_Stations"] = (
        df["Charging_Stations_Near_Home"]
        + df["Charging_Stations_Near_Work"]
    )

    df["Charging_Station_Difference"] = (
        df["Charging_Stations_Near_Home"]
        - df["Charging_Stations_Near_Work"]
    )

    df["Income_x_Concern"] = (
        df["Annual_Income_USD"]
        * df["Environmental_Concern_Level"]
    )

    df["Commute_x_Concern"] = (
        df["Daily_Commute_km"]
        * df["Environmental_Concern_Level"]
    )

    # Categorical interactions
    df["Subsidy_x_Concern"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Environmental_Concern_Level"].astype(str)
    )

    df["Subsidy_x_Anxiety"] = (
        df["Subsidy_Available"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    df["Concern_x_Anxiety"] = (
        df["Environmental_Concern_Level"].astype(str)
        + "_"
        + df["Range_Anxiety_Level"].astype(str)
    )

    income_band = pd.cut(
        df["Annual_Income_USD"],
        bins=[
            -np.inf,
            60000,
            80000,
            100000,
            120000,
            np.inf
        ],
        labels=[
            "Income_1",
            "Income_2",
            "Income_3",
            "Income_4",
            "Income_5"
        ]
    ).astype(str)

    df["Income_x_Subsidy"] = (
        income_band
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    df["HomeCharge_x_Subsidy"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + df["Subsidy_Available"].astype(str)
    )

    commute_band = pd.cut(
        df["Daily_Commute_km"],
        bins=[
            -np.inf,
            20,
            40,
            60,
            np.inf
        ],
        labels=[
            "Short",
            "Medium",
            "Long",
            "VeryLong"
        ]
    ).astype(str)

    df["HomeCharge_x_Commute"] = (
        df["Home_Charging_Possible"].astype(str)
        + "_"
        + commute_band
    )

    return df

In [20]:
train_fe = add_features(
    outer_train
)

valid_fe = add_features(
    outer_valid
)

In [21]:
X_train_art = (
    train_fe
    .drop(
        columns=[
            "id",
            "Will_Buy_EV"
        ]
    )
    .reset_index(drop=True)
)

In [22]:
X_train_art = (
    train_fe
    .drop(
        columns=[
            "id",
            "Will_Buy_EV"
        ]
    )
    .reset_index(drop=True)
)

X_valid_art = (
    valid_fe
    .drop(
        columns=[
            "id",
            "Will_Buy_EV"
        ]
    )
    .reset_index(drop=True)
)

X_train_art.shape, X_valid_art.shape

((534932, 24), (133733, 24))

In [23]:
X_train_art = pd.concat(
    [
        X_train_art,
        train_art_te,
        train_art_freq
    ],
    axis=1
)

X_valid_art = pd.concat(
    [
        X_valid_art,
        valid_art_te,
        valid_art_freq
    ],
    axis=1
)

In [24]:
categorical_cols = (
    X_train_art
    .select_dtypes(
        include=[
            "object",
            "str",
            "category"
        ]
    )
    .columns
    .tolist()
)

for col in categorical_cols:

    categories = pd.concat(
        [
            X_train_art[col].astype(str),
            X_valid_art[col].astype(str)
        ],
        ignore_index=True
    ).unique().tolist()

    X_train_art[col] = pd.Categorical(
        X_train_art[col].astype(str),
        categories=categories
    )

    X_valid_art[col] = pd.Categorical(
        X_valid_art[col].astype(str),
        categories=categories
    )

In [25]:
artifact_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [26]:
artifact_model.fit(
    X_train_art,
    y_outer_train.reset_index(
        drop=True
    ),

    eval_X=X_valid_art,
    eval_y=y_outer_valid.reset_index(
        drop=True
    ),

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.944814	valid_0's binary_logloss: 0.220955
[400]	valid_0's auc: 0.945165	valid_0's binary_logloss: 0.220189
[600]	valid_0's auc: 0.94525	valid_0's binary_logloss: 0.220006
[800]	valid_0's auc: 0.945254	valid_0's binary_logloss: 0.220003
Early stopping, best iteration is:
[847]	valid_0's auc: 0.945266	valid_0's binary_logloss: 0.219976
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [27]:
artifact_valid_pred = (
    artifact_model
    .predict_proba(
        X_valid_art
    )[:, 1]
)

artifact_auc = roc_auc_score(
    y_outer_valid,
    artifact_valid_pred
)

In [28]:
print(
    f"Raw LightGBM:           "
    f"{0.941669:.6f}"
)

print(
    f"Feature Engineering:    "
    f"{0.941785:.6f}"
)

print(
    f"FE + Target Encoding:   "
    f"{0.941771:.6f}"
)

print(
    f"FE + Original Rows:     "
    f"{0.941698:.6f}"
)

print(
    f"FE + Source Score:      "
    f"{0.941722:.6f}"
)

print(
    f"FE + Income Artifacts:  "
    f"{artifact_auc:.6f}"
)

print()

print(
    f"Gain vs FE: "
    f"{artifact_auc - 0.941785:+.6f}"
)

print(
    f"Best iteration: "
    f"{artifact_model.best_iteration_}"
)

Raw LightGBM:           0.941669
Feature Engineering:    0.941785
FE + Target Encoding:   0.941771
FE + Original Rows:     0.941698
FE + Source Score:      0.941722
FE + Income Artifacts:  0.945266

Gain vs FE: +0.003481
Best iteration: 847


In [29]:
artifact_importance = pd.DataFrame({
    "Feature":
        X_train_art.columns,

    "Importance":
        artifact_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

artifact_importance[
    artifact_importance[
        "Feature"
    ].str.contains(
        "TE_|FREQ_",
        regex=True
    )
]

,Feature,Importance
24,TE_Income_Exact,2482
25,TE_Income_Floor_50,1918
28,FREQ_Income_Exact,1758
29,FREQ_Income_Floor_50,1239
27,TE_Income_Floor_500,1219
26,TE_Income_Floor_250,1125
31,FREQ_Income_Floor_500,865
30,FREQ_Income_Floor_250,858


In [30]:
def create_artifact_features_three_way(
    train_df,
    valid_df,
    test_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    train_keys = make_artifact_keys(train_df)
    valid_keys = make_artifact_keys(valid_df)
    test_keys = make_artifact_keys(test_df)

    train_te = pd.DataFrame(index=train_df.index)
    valid_te = pd.DataFrame(index=valid_df.index)
    test_te = pd.DataFrame(index=test_df.index)

    train_freq = pd.DataFrame(index=train_df.index)
    valid_freq = pd.DataFrame(index=valid_df.index)
    test_freq = pd.DataFrame(index=test_df.index)

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        # -------------------------
        # Leak-free OOF TE
        # -------------------------
        oof_values = np.zeros(len(train_df))

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):
            mapping, global_mean = build_smoothed_mapping(
                train_keys[col].iloc[inner_train_idx],
                y_train.iloc[inner_train_idx],
                smoothing=smoothing
            )

            oof_values[inner_valid_idx] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_te[f"TE_{col}"] = oof_values

        # Mapping learned from whole OUTER training fold only
        full_mapping, full_global_mean = build_smoothed_mapping(
            train_keys[col],
            y_train,
            smoothing=smoothing
        )

        valid_te[f"TE_{col}"] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        test_te[f"TE_{col}"] = (
            test_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        # -------------------------
        # Frequency features
        # -------------------------
        counts = train_keys[col].value_counts()

        train_freq[f"FREQ_{col}"] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[f"FREQ_{col}"] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        test_freq[f"FREQ_{col}"] = (
            test_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    )

In [31]:
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions_artifact = np.zeros(len(train))
test_predictions_artifact = np.zeros(len(test))

fold_scores_artifact = []
best_iterations_artifact = []

In [32]:
for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(train, y),
    start=1
):

    print(f"\n{'=' * 60}")
    print(f"OUTER FOLD {fold}")
    print(f"{'=' * 60}")

    outer_train = train.iloc[train_idx].copy()
    outer_valid = train.iloc[valid_idx].copy()

    y_outer_train = y.iloc[train_idx].copy()
    y_outer_valid = y.iloc[valid_idx].copy()

    # --------------------------------
    # Leak-free synthetic artifacts
    # --------------------------------
    (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    ) = create_artifact_features_three_way(
        outer_train,
        outer_valid,
        test,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42 + fold
    )

    # --------------------------------
    # Ordinary engineered features
    # --------------------------------
    train_fe_fold = add_features(outer_train)
    valid_fe_fold = add_features(outer_valid)
    test_fe_fold = add_features(test)

    X_train_fold = (
        train_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_valid_fold = (
        valid_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_test_fold = (
        test_fe_fold
        .drop(columns=["id"])
        .reset_index(drop=True)
    )

    # --------------------------------
    # Append artifact features
    # --------------------------------
    X_train_fold = pd.concat(
        [
            X_train_fold,
            train_te,
            train_freq
        ],
        axis=1
    )

    X_valid_fold = pd.concat(
        [
            X_valid_fold,
            valid_te,
            valid_freq
        ],
        axis=1
    )

    X_test_fold = pd.concat(
        [
            X_test_fold,
            test_te,
            test_freq
        ],
        axis=1
    )

    # --------------------------------
    # Align categorical dtypes
    # --------------------------------
    categorical_cols = (
        X_train_fold
        .select_dtypes(
            include=["object", "str", "category"]
        )
        .columns
        .tolist()
    )

    for col in categorical_cols:

        categories = pd.concat(
            [
                X_train_fold[col].astype(str),
                X_valid_fold[col].astype(str),
                X_test_fold[col].astype(str)
            ],
            ignore_index=True
        ).unique().tolist()

        X_train_fold[col] = pd.Categorical(
            X_train_fold[col].astype(str),
            categories=categories
        )

        X_valid_fold[col] = pd.Categorical(
            X_valid_fold[col].astype(str),
            categories=categories
        )

        X_test_fold[col] = pd.Categorical(
            X_test_fold[col].astype(str),
            categories=categories
        )

    # --------------------------------
    # Model
    # --------------------------------
    model = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42 + fold,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_fold,
        y_outer_train.reset_index(drop=True),

        eval_X=X_valid_fold,
        eval_y=y_outer_valid.reset_index(drop=True),

        eval_metric="auc",

        callbacks=[
            early_stopping(
                150,
                first_metric_only=True
            ),
            log_evaluation(0)
        ]
    )

    valid_pred = model.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fold
    )[:, 1]

    oof_predictions_artifact[
        valid_idx
    ] = valid_pred

    test_predictions_artifact += (
        test_pred / 5
    )

    fold_auc = roc_auc_score(
        y_outer_valid,
        valid_pred
    )

    fold_scores_artifact.append(
        fold_auc
    )

    best_iterations_artifact.append(
        model.best_iteration_
    )

    print(
        f"Fold {fold} AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )


OUTER FOLD 1
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[842]	valid_0's auc: 0.94459	valid_0's binary_logloss: 0.22111
Evaluated only: auc
Fold 1 AUC: 0.944590
Best iteration: 842

OUTER FOLD 2
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[858]	valid_0's auc: 0.945256	valid_0's binary_logloss: 0.219798
Evaluated only: auc
Fold 2 AUC: 0.945256
Best iteration: 858

OUTER FOLD 3
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[938]	valid_0's auc: 0.946335	valid_0's binary_logloss: 0.217954
Evaluated only: auc
Fold 3 AUC: 0.946335
Best iteration: 938

OUTER FOLD 4
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[649]	valid_0's auc: 0.945939	valid_0's binary_logloss: 0.218623
Evaluated only: auc
Fold 4 AUC: 0.945939
Best iteration: 649

OUTER FOLD 5
Training until validation scores don't i

In [33]:
artifact_oof_auc = roc_auc_score(
    y,
    oof_predictions_artifact
)

print("\nFold scores:")

for fold, score in enumerate(
    fold_scores_artifact,
    start=1
):
    print(
        f"Fold {fold}: "
        f"{score:.6f}"
    )

print()

print(
    f"Mean fold AUC: "
    f"{np.mean(fold_scores_artifact):.6f}"
)

print(
    f"OOF AUC:       "
    f"{artifact_oof_auc:.6f}"
)

print(
    f"Std AUC:       "
    f"{np.std(fold_scores_artifact):.6f}"
)

print()

print(
    "Best iterations:",
    best_iterations_artifact
)


Fold scores:
Fold 1: 0.944590
Fold 2: 0.945256
Fold 3: 0.946335
Fold 4: 0.945939
Fold 5: 0.945468

Mean fold AUC: 0.945518
OOF AUC:       0.945511
Std AUC:       0.000596

Best iterations: [842, 858, 938, 649, 783]


In [34]:
artifact_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "artifact_lgb_oof":
        oof_predictions_artifact
})

artifact_oof_df.to_csv(
    "../outputs/predictions/"
    "lightgbm_income_artifact_oof.csv",
    index=False
)

In [35]:
artifact_test_df = pd.DataFrame({
    "id": test["id"],
    "artifact_lgb_prediction":
        test_predictions_artifact
})

artifact_test_df.to_csv(
    "../outputs/predictions/"
    "lightgbm_income_artifact_test.csv",
    index=False
)

In [36]:
cat_oof = pd.read_csv(
    "../outputs/predictions/catboost_oof.csv"
)

xgb_oof = pd.read_csv(
    "../outputs/predictions/xgboost_oof.csv"
)

fe_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_feature_engineered_oof.csv"
)

artifact_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_income_artifact_oof.csv"
)

In [37]:
assert cat_oof["id"].equals(xgb_oof["id"])
assert cat_oof["id"].equals(fe_oof["id"])
assert cat_oof["id"].equals(artifact_oof["id"])

print("OOF files aligned.")

OOF files aligned.


In [38]:
corr_matrix = pd.DataFrame({
    "CatBoost": cat_oof["catboost_oof"],
    "XGBoost": xgb_oof["xgboost_oof"],
    "FE_LightGBM": fe_oof["lgb_fe_oof"],
    "Artifact_LightGBM": artifact_oof["artifact_lgb_oof"]
}).corr()

corr_matrix

,CatBoost,XGBoost,FE_LightGBM,Artifact_LightGBM
CatBoost,1.000000,0.997973,0.997185,0.979795
XGBoost,0.997973,1.000000,0.997179,0.980184
FE_LightGBM,0.997185,0.997179,1.000000,0.981238
Artifact_LightGBM,0.979795,0.980184,0.981238,1.000000


In [39]:
y_ensemble = artifact_oof["target"]

cat_rank = cat_oof["catboost_oof"].rank(
    method="average",
    pct=True
)

xgb_rank = xgb_oof["xgboost_oof"].rank(
    method="average",
    pct=True
)

fe_rank = fe_oof["lgb_fe_oof"].rank(
    method="average",
    pct=True
)

artifact_rank = artifact_oof["artifact_lgb_oof"].rank(
    method="average",
    pct=True
)

In [40]:
old_ensemble_rank = (
    0.20 * cat_rank
    + 0.35 * fe_rank
    + 0.45 * xgb_rank
)

old_ensemble_auc = roc_auc_score(
    y_ensemble,
    old_ensemble_rank
)

old_ensemble_auc

0.9420459541460551

In [41]:
artifact_blend_results = []

for artifact_w in np.arange(0.0, 1.01, 0.05):

    old_w = 1.0 - artifact_w

    blended = (
        artifact_w * artifact_rank
        + old_w * old_ensemble_rank
    )

    auc = roc_auc_score(
        y_ensemble,
        blended
    )

    artifact_blend_results.append({
        "Artifact_Weight": artifact_w,
        "Old_Ensemble_Weight": old_w,
        "OOF_AUC": auc
    })

artifact_blend_results = pd.DataFrame(
    artifact_blend_results
)

artifact_blend_results.sort_values(
    "OOF_AUC",
    ascending=False
)

,Artifact_Weight,Old_Ensemble_Weight,OOF_AUC
18,0.90,0.10,0.945539
19,0.95,0.05,0.945536
17,0.85,0.15,0.945522
20,1.00,0.00,0.945511
16,0.80,0.20,0.945486
15,0.75,0.25,0.945429
14,0.70,0.30,0.945353
13,0.65,0.35,0.945256
12,0.60,0.40,0.945139
11,0.55,0.45,0.945001


In [42]:
best_artifact_blend = (
    artifact_blend_results
    .sort_values(
        "OOF_AUC",
        ascending=False
    )
    .iloc[0]
)

best_artifact_blend

Artifact_Weight        0.900000
Old_Ensemble_Weight    0.100000
OOF_AUC                0.945539
Name: 18, dtype: float64

In [43]:
pair_results = []

other_models = {
    "CatBoost": cat_rank,
    "XGBoost": xgb_rank,
    "FE_LightGBM": fe_rank
}

for model_name, model_rank in other_models.items():

    for artifact_w in np.arange(
        0.50,
        1.01,
        0.05
    ):

        other_w = 1 - artifact_w

        blended = (
            artifact_w * artifact_rank
            + other_w * model_rank
        )

        auc = roc_auc_score(
            y_ensemble,
            blended
        )

        pair_results.append({
            "Other_Model": model_name,
            "Artifact_Weight": artifact_w,
            "Other_Weight": other_w,
            "OOF_AUC": auc
        })

pair_results = pd.DataFrame(pair_results)

pair_results.sort_values(
    "OOF_AUC",
    ascending=False
).head(15)

,Other_Model,Artifact_Weight,Other_Weight,OOF_AUC
19,XGBoost,0.90,1.000000e-01,0.945544
8,CatBoost,0.90,1.000000e-01,0.945540
20,XGBoost,0.95,5.000000e-02,0.945539
9,CatBoost,0.95,5.000000e-02,0.945537
31,FE_LightGBM,0.95,5.000000e-02,0.945529
18,XGBoost,0.85,1.500000e-01,0.945528
30,FE_LightGBM,0.90,1.000000e-01,0.945525
7,CatBoost,0.85,1.500000e-01,0.945521
10,CatBoost,1.00,-4.440892e-16,0.945511
32,FE_LightGBM,1.00,-4.440892e-16,0.945511


In [44]:
print(f"Artifact model alone:    {artifact_oof_auc:.6f}")
print(f"Old ensemble:            {old_ensemble_auc:.6f}")

print()
print("Best artifact + old ensemble:")
print(best_artifact_blend)

print()
print("Best artifact pair blends:")
print(
    pair_results
    .sort_values("OOF_AUC", ascending=False)
    .head(10)
)

Artifact model alone:    0.945511
Old ensemble:            0.942046

Best artifact + old ensemble:
Artifact_Weight        0.900000
Old_Ensemble_Weight    0.100000
OOF_AUC                0.945539
Name: 18, dtype: float64

Best artifact pair blends:
    Other_Model  Artifact_Weight  Other_Weight   OOF_AUC
19      XGBoost             0.90  1.000000e-01  0.945544
8      CatBoost             0.90  1.000000e-01  0.945540
20      XGBoost             0.95  5.000000e-02  0.945539
9      CatBoost             0.95  5.000000e-02  0.945537
31  FE_LightGBM             0.95  5.000000e-02  0.945529
18      XGBoost             0.85  1.500000e-01  0.945528
30  FE_LightGBM             0.90  1.000000e-01  0.945525
7      CatBoost             0.85  1.500000e-01  0.945521
10     CatBoost             1.00 -4.440892e-16  0.945511
32  FE_LightGBM             1.00 -4.440892e-16  0.945511


In [45]:
def make_artifact_keys_v2(df):
    keys = pd.DataFrame(index=df.index)

    income = df["Annual_Income_USD"]
    commute = df["Daily_Commute_km"]

    # -------------------------
    # Income artifacts
    # -------------------------
    keys["Income_Exact"] = (
        income
        .round(0)
        .astype("Int64")
        .astype(str)
    )

    keys["Income_Floor_50"] = (
        np.floor(income / 50) * 50
    ).astype("Int64").astype(str)

    keys["Income_Floor_250"] = (
        np.floor(income / 250) * 250
    ).astype("Int64").astype(str)

    keys["Income_Floor_500"] = (
        np.floor(income / 500) * 500
    ).astype("Int64").astype(str)

    # -------------------------
    # Commute artifacts
    # -------------------------

    # Original values have roughly 0.1 km precision
    keys["Commute_Exact"] = (
        (commute * 10)
        .round()
        .astype("Int64")
        .astype(str)
    )

    keys["Commute_Floor_1"] = (
        np.floor(commute)
    ).astype("Int64").astype(str)

    keys["Commute_Floor_5"] = (
        np.floor(commute / 5) * 5
    ).astype("Int64").astype(str)

    keys["Commute_Floor_10"] = (
        np.floor(commute / 10) * 10
    ).astype("Int64").astype(str)

    return keys

In [46]:
artifact_keys_v2 = make_artifact_keys_v2(train)

artifact_keys_v2.nunique()

Income_Exact        13214
Income_Floor_50      2026
Income_Floor_250      533
Income_Floor_500      284
Commute_Exact         805
Commute_Floor_1        89
Commute_Floor_5        19
Commute_Floor_10       10
dtype: int64

In [47]:
def create_artifact_features_v2(
    train_df,
    valid_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    train_keys = make_artifact_keys_v2(train_df)
    valid_keys = make_artifact_keys_v2(valid_df)

    train_te = pd.DataFrame(index=train_df.index)
    valid_te = pd.DataFrame(index=valid_df.index)

    train_freq = pd.DataFrame(index=train_df.index)
    valid_freq = pd.DataFrame(index=valid_df.index)

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        # -------------------------
        # Leak-free target encoding
        # -------------------------
        oof_values = np.zeros(len(train_df))

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):
            mapping, global_mean = build_smoothed_mapping(
                train_keys[col].iloc[inner_train_idx],
                y_train.iloc[inner_train_idx],
                smoothing=smoothing
            )

            oof_values[inner_valid_idx] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_te[f"TE_{col}"] = oof_values

        full_mapping, full_global_mean = build_smoothed_mapping(
            train_keys[col],
            y_train,
            smoothing=smoothing
        )

        valid_te[f"TE_{col}"] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        # -------------------------
        # Frequency encoding
        # -------------------------
        counts = train_keys[col].value_counts()

        train_freq[f"FREQ_{col}"] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[f"FREQ_{col}"] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_te,
        valid_te,
        train_freq,
        valid_freq
    )

In [48]:
probe_train_idx, probe_valid_idx = train_test_split(
    np.arange(len(train)),
    test_size=0.20,
    random_state=42,
    stratify=y
)

probe_train = train.iloc[
    probe_train_idx
].copy()

probe_valid = train.iloc[
    probe_valid_idx
].copy()

y_probe_train = y.iloc[
    probe_train_idx
].copy()

y_probe_valid = y.iloc[
    probe_valid_idx
].copy()

In [49]:
(
    probe_train_te_v2,
    probe_valid_te_v2,
    probe_train_freq_v2,
    probe_valid_freq_v2
) = create_artifact_features_v2(
    probe_train,
    probe_valid,
    y_probe_train,
    smoothing=50,
    n_splits=5,
    random_state=42
)

In [50]:
probe_train_te_v2.shape, probe_valid_te_v2.shape

((534932, 8), (133733, 8))

In [51]:
probe_train_fe = add_features(
    probe_train
)

probe_valid_fe = add_features(
    probe_valid
)

In [52]:
X_probe_train_v2 = (
    probe_train_fe
    .drop(columns=["id", "Will_Buy_EV"])
    .reset_index(drop=True)
)

X_probe_valid_v2 = (
    probe_valid_fe
    .drop(columns=["id", "Will_Buy_EV"])
    .reset_index(drop=True)
)

In [53]:
X_probe_train_v2 = pd.concat(
    [
        X_probe_train_v2,
        probe_train_te_v2,
        probe_train_freq_v2
    ],
    axis=1
)

X_probe_valid_v2 = pd.concat(
    [
        X_probe_valid_v2,
        probe_valid_te_v2,
        probe_valid_freq_v2
    ],
    axis=1
)

In [54]:
X_probe_train_v2.shape, X_probe_valid_v2.shape

((534932, 40), (133733, 40))

In [55]:
categorical_cols_v2 = (
    X_probe_train_v2
    .select_dtypes(
        include=["object", "str", "category"]
    )
    .columns
    .tolist()
)

for col in categorical_cols_v2:

    categories = pd.concat(
        [
            X_probe_train_v2[col].astype(str),
            X_probe_valid_v2[col].astype(str)
        ],
        ignore_index=True
    ).unique().tolist()

    X_probe_train_v2[col] = pd.Categorical(
        X_probe_train_v2[col].astype(str),
        categories=categories
    )

    X_probe_valid_v2[col] = pd.Categorical(
        X_probe_valid_v2[col].astype(str),
        categories=categories
    )

In [56]:
artifact_v2_model = LGBMClassifier(
    objective="binary",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,

    subsample=0.9,
    colsample_bytree=0.9,

    reg_alpha=0.1,
    reg_lambda=1.0,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

In [57]:
artifact_v2_model.fit(
    X_probe_train_v2,
    y_probe_train.reset_index(drop=True),

    eval_X=X_probe_valid_v2,
    eval_y=y_probe_valid.reset_index(drop=True),

    eval_metric="auc",

    callbacks=[
        early_stopping(
            150,
            first_metric_only=True
        ),
        log_evaluation(200)
    ]
)

Training until validation scores don't improve for 150 rounds
[200]	valid_0's auc: 0.945018	valid_0's binary_logloss: 0.220552
[400]	valid_0's auc: 0.945316	valid_0's binary_logloss: 0.219882
[600]	valid_0's auc: 0.945374	valid_0's binary_logloss: 0.219756
[800]	valid_0's auc: 0.945378	valid_0's binary_logloss: 0.21974
[1000]	valid_0's auc: 0.94537	valid_0's binary_logloss: 0.219738
Early stopping, best iteration is:
[881]	valid_0's auc: 0.94539	valid_0's binary_logloss: 0.219717
Evaluated only: auc


,learning_rate,0.03
,n_estimators,3000
,objective,'binary'
,min_child_samples,50
,subsample,0.9
,colsample_bytree,0.9
,reg_alpha,0.1
,reg_lambda,1.0
,random_state,42
,n_jobs,-1
,verbosity,-1


In [58]:
artifact_v2_pred = (
    artifact_v2_model
    .predict_proba(
        X_probe_valid_v2
    )[:, 1]
)

artifact_v2_auc = roc_auc_score(
    y_probe_valid,
    artifact_v2_pred
)

In [59]:
print(f"FE only:                 {0.941785:.6f}")
print(f"Income artifacts V1:     {0.945266:.6f}")
print(f"Income + commute V2:     {artifact_v2_auc:.6f}")

print()

print(
    f"Gain vs V1: "
    f"{artifact_v2_auc - 0.945266:+.6f}"
)

print(
    f"Best iteration: "
    f"{artifact_v2_model.best_iteration_}"
)

FE only:                 0.941785
Income artifacts V1:     0.945266
Income + commute V2:     0.945390

Gain vs V1: +0.000124
Best iteration: 881


In [60]:
artifact_v2_importance = pd.DataFrame({
    "Feature": X_probe_train_v2.columns,
    "Importance": artifact_v2_model.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
)

artifact_v2_importance[
    artifact_v2_importance[
        "Feature"
    ].str.contains(
        "Commute",
        regex=False
    )
]

,Feature,Importance
28,TE_Commute_Exact,1282
29,TE_Commute_Floor_1,1257
36,FREQ_Commute_Exact,853
17,Commute_x_Concern,656
2,Daily_Commute_km,519
37,FREQ_Commute_Floor_1,506
30,TE_Commute_Floor_5,506
23,HomeCharge_x_Commute,328
31,TE_Commute_Floor_10,320
38,FREQ_Commute_Floor_5,114


In [61]:
artifact_test = pd.read_csv(
    "../outputs/predictions/lightgbm_income_artifact_test.csv"
)

artifact_test.head()

,id,artifact_lgb_prediction
0,668665,0.028643
1,668666,0.020016
2,668667,0.004196
3,668668,0.002410
4,668669,0.028581


In [62]:
submission_005 = pd.DataFrame({
    "id": artifact_test["id"],
    "Will_Buy_EV": artifact_test["artifact_lgb_prediction"]
})

In [63]:
submission_005 = pd.DataFrame({
    "id": artifact_test["id"],
    "Will_Buy_EV": artifact_test["artifact_lgb_prediction"]
})

In [64]:
submission_005.shape

(286571, 2)

In [65]:
submission_005.isna().sum()

id             0
Will_Buy_EV    0
dtype: int64

In [66]:
submission_005["id"].equals(test["id"])

True

In [67]:
submission_005.to_csv(
    "../outputs/submissions/submission_005_income_artifact_lgbm.csv",
    index=False
)

## Experiment V2 — Income + Commute Synthetic Artifacts

V1 showed a strong synthetic-generation artifact in `Annual_Income_USD`.

V2 tests whether `Daily_Commute_km` contains a similar hidden generation pattern.

New commute keys:
- Exact 0.1 km
- Floor 1 km
- Floor 5 km
- Floor 10 km

Each key receives:
- Leak-free target encoding
- Frequency encoding

Baseline to beat:
- Income Artifact V1 OOF AUC: 0.945511
- Public LB: 0.94580

In [68]:
def add_commute_artifact_keys(df):
    df = df.copy()

    commute = df["Daily_Commute_km"]

    # Exact recorded value (dataset resolution = 0.1 km)
    df["Commute_exact_key"] = commute.round(1).astype(str)

    # Quantised / floored representations
    df["Commute_floor_1_key"] = np.floor(commute / 1) * 1
    df["Commute_floor_5_key"] = np.floor(commute / 5) * 5
    df["Commute_floor_10_key"] = np.floor(commute / 10) * 10

    return df

In [69]:
train_v2 = add_commute_artifact_keys(train)
test_v2 = add_commute_artifact_keys(test)

commute_keys = [
    "Commute_exact_key",
    "Commute_floor_1_key",
    "Commute_floor_5_key",
    "Commute_floor_10_key",
]

train_v2[commute_keys].head()

,Commute_exact_key,Commute_floor_1_key,Commute_floor_5_key,Commute_floor_10_key
0,23.4,23.0,20.0,20.0
1,5.0,5.0,5.0,0.0
2,36.8,36.0,35.0,30.0
3,23.7,23.0,20.0,20.0
4,50.8,50.0,50.0,50.0


In [70]:
for col in commute_keys:
    print(
        f"{col:22s} "
        f"train unique = {train_v2[col].nunique():5d} | "
        f"test unique = {test_v2[col].nunique():5d}"
    )

Commute_exact_key      train unique =   805 | test unique =   781
Commute_floor_1_key    train unique =    89 | test unique =    90
Commute_floor_5_key    train unique =    19 | test unique =    20
Commute_floor_10_key   train unique =    10 | test unique =    11


In [71]:
for col in commute_keys:
    train_values = set(train_v2[col].unique())
    test_values = set(test_v2[col].unique())

    unseen_test = test_values - train_values

    counts = train_v2[col].value_counts()

    print(f"\n{col}")
    print("-" * 50)
    print("Train unique:", len(train_values))
    print("Test unique: ", len(test_values))
    print("Unseen in test:", len(unseen_test))
    print("Min group size:", counts.min())
    print("Median group size:", counts.median())
    print("Mean group size:", round(counts.mean(), 2))
    print("Max group size:", counts.max())


Commute_exact_key
--------------------------------------------------
Train unique: 805
Test unique:  781
Unseen in test: 28
Min group size: 1
Median group size: 576.0
Mean group size: 830.64
Max group size: 144280

Commute_floor_1_key
--------------------------------------------------
Train unique: 89
Test unique:  90
Unseen in test: 1
Min group size: 1
Median group size: 5429.0
Mean group size: 7513.09
Max group size: 145199

Commute_floor_5_key
--------------------------------------------------
Train unique: 19
Test unique:  20
Unseen in test: 1
Min group size: 5
Median group size: 28198.0
Mean group size: 35192.89
Max group size: 145959

Commute_floor_10_key
--------------------------------------------------
Train unique: 10
Test unique:  11
Unseen in test: 1
Min group size: 46
Median group size: 68541.0
Mean group size: 66866.5
Max group size: 145959


In [73]:
commute_diag = train_v2[["Commute_exact_key"]].copy()
commute_diag["target"] = np.asarray(y)

exact_commute_stats = (
    commute_diag
    .groupby("Commute_exact_key", observed=True)
    .agg(
        count=("target", "size"),
        purchase_rate=("target", "mean"),
    )
    .sort_values("count", ascending=False)
)

exact_commute_stats.head(20)

,count,purchase_rate
Commute_exact_key,,
5.0,144280,0.184426
46.6,2756,0.182511
48.5,2720,0.143750
33.0,2562,0.153396
53.3,2546,0.160644
30.1,2370,0.152321
47.1,2347,0.158926
52.2,2279,0.161913
32.7,2242,0.187333


In [74]:
print("Overall purchase rate:", y.mean())

print("\nLargest exact commute groups:")
display(exact_commute_stats.head(20))

print("\nSmallest exact commute groups:")
display(exact_commute_stats.tail(20))

Overall purchase rate: 0.17464500160768098

Largest exact commute groups:


,count,purchase_rate
Commute_exact_key,,
5.0,144280,0.184426
46.6,2756,0.182511
48.5,2720,0.143750
33.0,2562,0.153396
53.3,2546,0.160644
30.1,2370,0.152321
47.1,2347,0.158926
52.2,2279,0.161913
32.7,2242,0.187333



Smallest exact commute groups:


,count,purchase_rate
Commute_exact_key,,
82.7,1,0.0
82.5,1,1.0
81.3,1,0.0
84.4,1,0.0
87.0,1,0.0
86.8,1,0.0
86.5,1,0.0
86.3,1,0.0
86.0,1,0.0


In [75]:
print("Overall purchase rate:", y.mean())

print("\nLargest exact commute groups:")
display(exact_commute_stats.head(20))

print("\nSmallest exact commute groups:")
display(exact_commute_stats.tail(20))

Overall purchase rate: 0.17464500160768098

Largest exact commute groups:


,count,purchase_rate
Commute_exact_key,,
5.0,144280,0.184426
46.6,2756,0.182511
48.5,2720,0.143750
33.0,2562,0.153396
53.3,2546,0.160644
30.1,2370,0.152321
47.1,2347,0.158926
52.2,2279,0.161913
32.7,2242,0.187333



Smallest exact commute groups:


,count,purchase_rate
Commute_exact_key,,
82.7,1,0.0
82.5,1,1.0
81.3,1,0.0
84.4,1,0.0
87.0,1,0.0
86.8,1,0.0
86.5,1,0.0
86.3,1,0.0
86.0,1,0.0


In [76]:
unseen_exact = sorted(
    set(test_v2["Commute_exact_key"]) -
    set(train_v2["Commute_exact_key"])
)

print("Number of unseen exact commute values:", len(unseen_exact))
print(unseen_exact)

Number of unseen exact commute values: 28
['103.2', '103.9', '6.2', '6.7', '72.2', '72.4', '72.8', '73.2', '77.6', '77.7', '80.1', '80.8', '82.3', '82.6', '83.2', '85.6', '86.4', '86.7', '86.9', '87.6', '87.8', '88.1', '89.5', '90.1', '92.3', '92.6', '96.9', '98.5']


In [78]:
%whos function

Variable                             Type        Data/Info
----------------------------------------------------------
add_commute_artifact_keys            function    <function add_commute_art<...>t_keys at 0x767e3e5267a0>
add_features                         function    <function add_features at 0x767eb81eaac0>
build_smoothed_mapping               function    <function build_smoothed_<...>apping at 0x767ebaae05e0>
create_artifact_encodings            function    <function create_artifact<...>odings at 0x767ebaae2160>
create_artifact_features_three_way   function    <function create_artifact<...>ee_way at 0x767e4cadb600>
create_artifact_features_v2          function    <function create_artifact<...>res_v2 at 0x767e185dc400>
create_frequency_features            function    <function create_frequenc<...>atures at 0x767e4cacc860>
early_stopping                       function    <function early_stopping at 0x767e59bb5300>
log_evaluation                       function    <function log_evalu

In [79]:
print([
    c for c in train.columns
    if "income" in c.lower()
])

print([
    c for c in train_v2.columns
    if "income" in c.lower() or "commute" in c.lower()
])

['Annual_Income_USD']
['Annual_Income_USD', 'Daily_Commute_km', 'Commute_exact_key', 'Commute_floor_1_key', 'Commute_floor_5_key', 'Commute_floor_10_key']


In [80]:
import inspect

print("=== make_artifact_keys_v2 ===")
print(inspect.getsource(make_artifact_keys_v2))

print("\n=== create_artifact_features_v2 ===")
print(inspect.getsource(create_artifact_features_v2))

print("\n=== create_artifact_encodings ===")
print(inspect.getsource(create_artifact_encodings))

=== make_artifact_keys_v2 ===
def make_artifact_keys_v2(df):
    keys = pd.DataFrame(index=df.index)

    income = df["Annual_Income_USD"]
    commute = df["Daily_Commute_km"]

    # -------------------------
    # Income artifacts
    # -------------------------
    keys["Income_Exact"] = (
        income
        .round(0)
        .astype("Int64")
        .astype(str)
    )

    keys["Income_Floor_50"] = (
        np.floor(income / 50) * 50
    ).astype("Int64").astype(str)

    keys["Income_Floor_250"] = (
        np.floor(income / 250) * 250
    ).astype("Int64").astype(str)

    keys["Income_Floor_500"] = (
        np.floor(income / 500) * 500
    ).astype("Int64").astype(str)

    # -------------------------
    # Commute artifacts
    # -------------------------

    # Original values have roughly 0.1 km precision
    keys["Commute_Exact"] = (
        (commute * 10)
        .round()
        .astype("Int64")
        .astype(str)
    )

    keys["Commute_Floor_1"] = (
        np.fl

In [81]:
print("=== create_frequency_features ===")
print(inspect.getsource(create_frequency_features))

print("\n=== create_artifact_features_three_way ===")
print(inspect.getsource(create_artifact_features_three_way))

=== create_frequency_features ===
def create_frequency_features(
    train_df,
    valid_df
):
    train_df = train_df.reset_index(
        drop=True
    )

    valid_df = valid_df.reset_index(
        drop=True
    )

    train_keys = make_artifact_keys(
        train_df
    )

    valid_keys = make_artifact_keys(
        valid_df
    )

    train_freq = pd.DataFrame(
        index=train_df.index
    )

    valid_freq = pd.DataFrame(
        index=valid_df.index
    )

    for col in train_keys.columns:

        counts = (
            train_keys[col]
            .value_counts()
        )

        train_freq[
            f"FREQ_{col}"
        ] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[
            f"FREQ_{col}"
        ] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_freq,
        valid_freq
    )


=

In [82]:
def create_artifact_features_v2_three_way(
    train_df,
    valid_df,
    test_df,
    y_train,
    smoothing=50,
    n_splits=5,
    random_state=42
):
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    y_train = y_train.reset_index(drop=True)

    # V2 = income + commute artifact keys
    train_keys = make_artifact_keys_v2(train_df)
    valid_keys = make_artifact_keys_v2(valid_df)
    test_keys = make_artifact_keys_v2(test_df)

    train_te = pd.DataFrame(index=train_df.index)
    valid_te = pd.DataFrame(index=valid_df.index)
    test_te = pd.DataFrame(index=test_df.index)

    train_freq = pd.DataFrame(index=train_df.index)
    valid_freq = pd.DataFrame(index=valid_df.index)
    test_freq = pd.DataFrame(index=test_df.index)

    inner_cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    for col in train_keys.columns:

        # --------------------------------
        # Leak-free OOF target encoding
        # --------------------------------
        oof_values = np.zeros(len(train_df))

        for inner_train_idx, inner_valid_idx in inner_cv.split(
            train_df,
            y_train
        ):
            mapping, global_mean = build_smoothed_mapping(
                train_keys[col].iloc[inner_train_idx],
                y_train.iloc[inner_train_idx],
                smoothing=smoothing
            )

            oof_values[inner_valid_idx] = (
                train_keys[col]
                .iloc[inner_valid_idx]
                .astype(str)
                .map(mapping)
                .fillna(global_mean)
                .to_numpy()
            )

        train_te[f"TE_{col}"] = oof_values

        # Mapping learned ONLY from whole outer-training fold
        full_mapping, full_global_mean = build_smoothed_mapping(
            train_keys[col],
            y_train,
            smoothing=smoothing
        )

        valid_te[f"TE_{col}"] = (
            valid_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        test_te[f"TE_{col}"] = (
            test_keys[col]
            .astype(str)
            .map(full_mapping)
            .fillna(full_global_mean)
            .to_numpy()
        )

        # --------------------------------
        # Frequency encoding
        # --------------------------------
        counts = train_keys[col].value_counts()

        train_freq[f"FREQ_{col}"] = (
            train_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        valid_freq[f"FREQ_{col}"] = (
            valid_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

        test_freq[f"FREQ_{col}"] = (
            test_keys[col]
            .map(counts)
            .fillna(0)
            .to_numpy()
        )

    return (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    )

In [83]:
print(make_artifact_keys_v2(train).columns.tolist())

['Income_Exact', 'Income_Floor_50', 'Income_Floor_250', 'Income_Floor_500', 'Commute_Exact', 'Commute_Floor_1', 'Commute_Floor_5', 'Commute_Floor_10']


In [85]:
for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(train, y),
    start=1
):

    print(f"\n{'=' * 60}")
    print(f"OUTER FOLD {fold}")
    print(f"{'=' * 60}")

    outer_train = train.iloc[train_idx].copy()
    outer_valid = train.iloc[valid_idx].copy()

    y_outer_train = y.iloc[train_idx].copy()
    y_outer_valid = y.iloc[valid_idx].copy()

    # --------------------------------
    # Leak-free synthetic artifacts
    # --------------------------------
    (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    ) = create_artifact_features_v2_three_way(
        outer_train,
        outer_valid,
        test,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42 + fold
    )

    # --------------------------------
    # Ordinary engineered features
    # --------------------------------
    train_fe_fold = add_features(outer_train)
    valid_fe_fold = add_features(outer_valid)
    test_fe_fold = add_features(test)

    X_train_fold = (
        train_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_valid_fold = (
        valid_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_test_fold = (
        test_fe_fold
        .drop(columns=["id"])
        .reset_index(drop=True)
    )

    # --------------------------------
    # Append artifact features
    # --------------------------------
    X_train_fold = pd.concat(
        [
            X_train_fold,
            train_te,
            train_freq
        ],
        axis=1
    )

    X_valid_fold = pd.concat(
        [
            X_valid_fold,
            valid_te,
            valid_freq
        ],
        axis=1
    )

    X_test_fold = pd.concat(
        [
            X_test_fold,
            test_te,
            test_freq
        ],
        axis=1
    )

    # --------------------------------
    # Align categorical dtypes
    # --------------------------------
    categorical_cols = (
        X_train_fold
        .select_dtypes(
            include=["object", "str", "category"]
        )
        .columns
        .tolist()
    )

    for col in categorical_cols:

        categories = pd.concat(
            [
                X_train_fold[col].astype(str),
                X_valid_fold[col].astype(str),
                X_test_fold[col].astype(str)
            ],
            ignore_index=True
        ).unique().tolist()

        X_train_fold[col] = pd.Categorical(
            X_train_fold[col].astype(str),
            categories=categories
        )

        X_valid_fold[col] = pd.Categorical(
            X_valid_fold[col].astype(str),
            categories=categories
        )

        X_test_fold[col] = pd.Categorical(
            X_test_fold[col].astype(str),
            categories=categories
        )

    # --------------------------------
    # Model
    # --------------------------------
    model = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42 + fold,
        n_jobs=-1,
        verbosity=-1
    )

    model.fit(
        X_train_fold,
        y_outer_train.reset_index(drop=True),

        eval_X=X_valid_fold,
        eval_y=y_outer_valid.reset_index(drop=True),

        eval_metric="auc",

        callbacks=[
            early_stopping(
                150,
                first_metric_only=True
            ),
            log_evaluation(0)
        ]
    )

    valid_pred = model.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fold
    )[:, 1]

    oof_predictions_artifact[
        valid_idx
    ] = valid_pred

    test_predictions_artifact += (
        test_pred / 5
    )

    fold_auc = roc_auc_score(
        y_outer_valid,
        valid_pred
    )

    fold_scores_artifact.append(
        fold_auc
    )

    best_iterations_artifact.append(
        model.best_iteration_
    )

    print(
        f"Fold {fold} AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )


OUTER FOLD 1
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[882]	valid_0's auc: 0.944651	valid_0's binary_logloss: 0.221034
Evaluated only: auc
Fold 1 AUC: 0.944651
Best iteration: 882

OUTER FOLD 2
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[785]	valid_0's auc: 0.945348	valid_0's binary_logloss: 0.219673
Evaluated only: auc
Fold 2 AUC: 0.945348
Best iteration: 785

OUTER FOLD 3
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[703]	valid_0's auc: 0.946441	valid_0's binary_logloss: 0.217736
Evaluated only: auc
Fold 3 AUC: 0.946441
Best iteration: 703

OUTER FOLD 4
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[646]	valid_0's auc: 0.945979	valid_0's binary_logloss: 0.218563
Evaluated only: auc
Fold 4 AUC: 0.945979
Best iteration: 646

OUTER FOLD 5
Training until validation scores don't

In [87]:
for name, value in globals().copy().items():
    try:
        if isinstance(value, (np.ndarray, pd.Series)):
            if len(value) in [len(train), len(test)]:
                print(
                    f"{name:35s}",
                    type(value).__name__,
                    value.shape if hasattr(value, "shape") else len(value)
                )
    except:
        pass

y                                   Series (668665,)
oof_predictions_artifact            ndarray (668665,)
test_predictions_artifact           ndarray (286571,)
test_pred                           ndarray (286571,)
y_ensemble                          Series (668665,)
cat_rank                            Series (668665,)
xgb_rank                            Series (668665,)
fe_rank                             Series (668665,)
artifact_rank                       Series (668665,)
old_ensemble_rank                   Series (668665,)
blended                             Series (668665,)
model_rank                          Series (668665,)


In [88]:
oof_predictions_v2 = oof_predictions_artifact.copy()
test_predictions_v2 = test_predictions_artifact.copy()

In [89]:
v2_oof_auc = roc_auc_score(y, oof_predictions_v2)

print(f"V2 OOF AUC: {v2_oof_auc:.6f}")
print(f"V1 OOF AUC: {0.945511:.6f}")
print(f"Difference: {v2_oof_auc - 0.945511:+.6f}")

V2 OOF AUC: 0.945599
V1 OOF AUC: 0.945511
Difference: +0.000088


In [90]:
fold_scores_v2 = [
    0.944651,
    0.945348,
    0.946441,
    0.945979,
    0.945624
]

print("Mean fold AUC:", np.mean(fold_scores_v2))
print("Fold std:", np.std(fold_scores_v2))

Mean fold AUC: 0.9456085999999999
Fold std: 0.0006025335177398773


In [91]:
v2_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "v2_oof": oof_predictions_v2
})

v2_test_df = pd.DataFrame({
    "id": test["id"],
    "v2_prediction": test_predictions_v2
})

v2_oof_df.to_csv(
    "../outputs/predictions/income_commute_v2_oof.csv",
    index=False
)

v2_test_df.to_csv(
    "../outputs/predictions/income_commute_v2_test.csv",
    index=False
)

print("V2 predictions saved.")

V2 predictions saved.


In [92]:
from pathlib import Path

prediction_dir = Path("../outputs/predictions")

for file in sorted(prediction_dir.glob("*")):
    print(file.name)

catboost_oof.csv
catboost_test.csv
income_commute_v2_oof.csv
income_commute_v2_test.csv
lightgbm_feature_engineered_oof.csv
lightgbm_feature_engineered_test.csv
lightgbm_income_artifact_oof.csv
lightgbm_income_artifact_test.csv
lightgbm_oof.csv
lightgbm_test.csv
xgboost_oof.csv
xgboost_test.csv


In [93]:
v1_oof = pd.read_csv(
    "../outputs/predictions/lightgbm_income_artifact_oof.csv"
)

v2_oof = pd.read_csv(
    "../outputs/predictions/income_commute_v2_oof.csv"
)

print(v1_oof.columns.tolist())
print(v2_oof.columns.tolist())
print(v1_oof.shape, v2_oof.shape)

['id', 'target', 'artifact_lgb_oof']
['id', 'target', 'v2_oof']
(668665, 3) (668665, 3)


In [94]:
from scipy.stats import spearmanr

v1_pred = v1_oof["artifact_lgb_oof"].to_numpy()
v2_pred = v2_oof["v2_oof"].to_numpy()
target = v1_oof["target"].to_numpy()

print("ID alignment:", v1_oof["id"].equals(v2_oof["id"]))
print("Target alignment:", np.array_equal(
    v1_oof["target"].to_numpy(),
    v2_oof["target"].to_numpy()
))

v1_auc = roc_auc_score(target, v1_pred)
v2_auc = roc_auc_score(target, v2_pred)

pearson_corr = np.corrcoef(v1_pred, v2_pred)[0, 1]
spearman_corr = spearmanr(v1_pred, v2_pred).statistic

print(f"\nV1 AUC: {v1_auc:.9f}")
print(f"V2 AUC: {v2_auc:.9f}")
print(f"Pearson correlation: {pearson_corr:.6f}")
print(f"Spearman correlation: {spearman_corr:.6f}")

ID alignment: True
Target alignment: True

V1 AUC: 0.945510893
V2 AUC: 0.945599254
Pearson correlation: 0.998536
Spearman correlation: 0.998694


In [95]:
probability_results = []

for v2_weight in np.arange(0.0, 1.01, 0.05):
    v1_weight = 1.0 - v2_weight

    blended_pred = (
        v1_weight * v1_pred +
        v2_weight * v2_pred
    )

    auc = roc_auc_score(target, blended_pred)

    probability_results.append({
        "V1_weight": v1_weight,
        "V2_weight": v2_weight,
        "AUC": auc
    })

probability_results = pd.DataFrame(probability_results)

probability_results.sort_values(
    "AUC",
    ascending=False
).head(10)

,V1_weight,V2_weight,AUC
13,0.35,0.65,0.945645
12,0.40,0.60,0.945645
14,0.30,0.70,0.945644
11,0.45,0.55,0.945643
15,0.25,0.75,0.945640
10,0.50,0.50,0.945639
16,0.20,0.80,0.945636
9,0.55,0.45,0.945634
17,0.15,0.85,0.945629
8,0.60,0.40,0.945627


In [96]:
v1_rank = pd.Series(v1_pred).rank(pct=True).to_numpy()
v2_rank = pd.Series(v2_pred).rank(pct=True).to_numpy()

rank_results = []

for v2_weight in np.arange(0.0, 1.01, 0.05):
    v1_weight = 1.0 - v2_weight

    blended_rank = (
        v1_weight * v1_rank +
        v2_weight * v2_rank
    )

    auc = roc_auc_score(target, blended_rank)

    rank_results.append({
        "V1_weight": v1_weight,
        "V2_weight": v2_weight,
        "AUC": auc
    })

rank_results = pd.DataFrame(rank_results)

rank_results.sort_values(
    "AUC",
    ascending=False
).head(10)

,V1_weight,V2_weight,AUC
13,0.35,0.65,0.945646
12,0.40,0.60,0.945646
14,0.30,0.70,0.945644
11,0.45,0.55,0.945644
15,0.25,0.75,0.945641
10,0.50,0.50,0.945640
16,0.20,0.80,0.945636
9,0.55,0.45,0.945635
17,0.15,0.85,0.945629
8,0.60,0.40,0.945628


In [97]:
fine_rank_results = []

for v2_weight in np.arange(0.50, 0.801, 0.01):
    v1_weight = 1.0 - v2_weight

    blended_rank = (
        v1_weight * v1_rank +
        v2_weight * v2_rank
    )

    auc = roc_auc_score(target, blended_rank)

    fine_rank_results.append({
        "V1_weight": v1_weight,
        "V2_weight": v2_weight,
        "AUC": auc
    })

fine_rank_results = pd.DataFrame(fine_rank_results)

fine_rank_results.sort_values(
    "AUC",
    ascending=False
).head(15)

,V1_weight,V2_weight,AUC
13,0.37,0.63,0.945646
12,0.38,0.62,0.945646
14,0.36,0.64,0.945646
11,0.39,0.61,0.945646
15,0.35,0.65,0.945646
10,0.40,0.60,0.945646
16,0.34,0.66,0.945646
9,0.41,0.59,0.945645
17,0.33,0.67,0.945645
8,0.42,0.58,0.945645


In [98]:
fine_prob_results = []

for v2_weight in np.arange(0.50, 0.801, 0.01):
    v1_weight = 1.0 - v2_weight

    blended_pred = (
        v1_weight * v1_pred +
        v2_weight * v2_pred
    )

    auc = roc_auc_score(target, blended_pred)

    fine_prob_results.append({
        "V1_weight": v1_weight,
        "V2_weight": v2_weight,
        "AUC": auc
    })

fine_prob_results = pd.DataFrame(fine_prob_results)

fine_prob_results.sort_values(
    "AUC",
    ascending=False
).head(15)

,V1_weight,V2_weight,AUC
14,0.36,0.64,0.945645
13,0.37,0.63,0.945645
15,0.35,0.65,0.945645
12,0.38,0.62,0.945645
16,0.34,0.66,0.945645
11,0.39,0.61,0.945645
10,0.40,0.60,0.945645
17,0.33,0.67,0.945645
9,0.41,0.59,0.945644
18,0.32,0.68,0.945644


In [99]:
cv_check = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# We'll start with 35% V1 + 65% V2 rank blend.
# We may change this after the fine search.
candidate_blend = (
    0.35 * v1_rank +
    0.65 * v2_rank
)

print("Fold comparison")
print("-" * 70)

for fold, (_, valid_idx) in enumerate(
    cv_check.split(np.zeros(len(target)), target),
    1
):
    auc_v1 = roc_auc_score(
        target[valid_idx],
        v1_pred[valid_idx]
    )

    auc_v2 = roc_auc_score(
        target[valid_idx],
        v2_pred[valid_idx]
    )

    auc_blend = roc_auc_score(
        target[valid_idx],
        candidate_blend[valid_idx]
    )

    print(
        f"Fold {fold}: "
        f"V1={auc_v1:.6f} | "
        f"V2={auc_v2:.6f} | "
        f"Blend={auc_blend:.6f} | "
        f"Blend-V1={auc_blend-auc_v1:+.6f}"
    )

Fold comparison
----------------------------------------------------------------------
Fold 1: V1=0.944590 | V2=0.944651 | Blend=0.944705 | Blend-V1=+0.000115
Fold 2: V1=0.945256 | V2=0.945348 | Blend=0.945398 | Blend-V1=+0.000142
Fold 3: V1=0.946335 | V2=0.946441 | Blend=0.946479 | Blend-V1=+0.000145
Fold 4: V1=0.945939 | V2=0.945979 | Blend=0.946040 | Blend-V1=+0.000101
Fold 5: V1=0.945468 | V2=0.945624 | Blend=0.945645 | Blend-V1=+0.000177


In [100]:
v1_test = pd.read_csv(
    "../outputs/predictions/lightgbm_income_artifact_test.csv"
)

v2_test = pd.read_csv(
    "../outputs/predictions/income_commute_v2_test.csv"
)

print(v1_test.columns.tolist())
print(v2_test.columns.tolist())
print(v1_test.shape, v2_test.shape)

print(
    "ID alignment:",
    v1_test["id"].equals(v2_test["id"])
)

['id', 'artifact_lgb_prediction']
['id', 'v2_prediction']
(286571, 2) (286571, 2)
ID alignment: True


In [101]:
# Test predictions
v1_test_pred = v1_test["artifact_lgb_prediction"].to_numpy()
v2_test_pred = v2_test["v2_prediction"].to_numpy()

# Convert each model's test predictions to percentile ranks
v1_test_rank = pd.Series(v1_test_pred).rank(pct=True).to_numpy()
v2_test_rank = pd.Series(v2_test_pred).rank(pct=True).to_numpy()

# 35% V1 + 65% V2 rank blend
test_blend_rank = (
    0.35 * v1_test_rank +
    0.65 * v2_test_rank
)

submission_006 = pd.DataFrame({
    "id": v1_test["id"],
    "Will_Buy_EV": test_blend_rank
})

print(submission_006.head())
print(submission_006.shape)
print(submission_006.isna().sum())
print("ID alignment:", submission_006["id"].equals(test["id"]))
print(
    "Prediction range:",
    submission_006["Will_Buy_EV"].min(),
    submission_006["Will_Buy_EV"].max()
)

       id  Will_Buy_EV
0  668665     0.565901
1  668666     0.520990
2  668667     0.304651
3  668668     0.249090
4  668669     0.556334
(286571, 2)
id             0
Will_Buy_EV    0
dtype: int64
ID alignment: True
Prediction range: 3.489536624431642e-06 1.0


In [102]:
submission_006.to_csv(
    "../outputs/submissions/submission_006_income_commute_rank_blend.csv",
    index=False
)

print("Submission #006 saved.")

Submission #006 saved.


## Experiment V2 Results — Income + Commute Artifacts

### V2 LightGBM
- Income artifact features retained
- Added commute artifact keys:
  - Exact 0.1 km
  - Floor 1 km
  - Floor 5 km
  - Floor 10 km
- Leak-free nested 5-fold target encoding
- Frequency encoding
- Same LightGBM setup as V1

### Outer-fold AUC
- Fold 1: 0.944651
- Fold 2: 0.945348
- Fold 3: 0.946441
- Fold 4: 0.945979
- Fold 5: 0.945624

Mean fold AUC: 0.945609  
OOF AUC: 0.945599

V1 OOF: 0.945511  
V2 improvement: +0.000088

### V1 / V2 Blend
Prediction correlation:
- Pearson: 0.998536
- Spearman: 0.998694

35% V1 + 65% V2 rank blend:
- OOF AUC: ~0.945645
- Improved all 5 validation folds

Submission #006:
`submission_006_income_commute_rank_blend.csv`

Public LB: **0.94589**

Previous best:
Submission #005 Public LB: 0.94580

Public LB improvement: +0.00009

In [103]:
spike_mask = train["Daily_Commute_km"].eq(5.0)

print("Rows at exactly 5.0 km:", spike_mask.sum())
print("Fraction of train:", spike_mask.mean())

print(
    "Purchase rate at 5.0:",
    y[spike_mask].mean()
)

print(
    "Purchase rate elsewhere:",
    y[~spike_mask].mean()
)

Rows at exactly 5.0 km: 144280
Fraction of train: 0.21577321977372824
Purchase rate at 5.0: 0.18442611588577765
Purchase rate elsewhere: 0.17195381256138143


In [104]:
spike_analysis = (
    train.assign(
        target_numeric=y,
        Commute_is_5=(train["Daily_Commute_km"] == 5.0)
    )
    .groupby("Commute_is_5")
    .agg(
        rows=("target_numeric", "size"),
        purchase_rate=("target_numeric", "mean"),
        income_mean=("Annual_Income_USD", "mean"),
        age_mean=("Age", "mean"),
        environmental_mean=("Environmental_Concern_Level", "mean")
    )
)

spike_analysis

,rows,purchase_rate,income_mean,age_mean,environmental_mean
Commute_is_5,,,,,
False,524385,0.171954,84936.640754,47.004180,2.933646
True,144280,0.184426,84160.947803,47.166343,2.942133


In [105]:
digit_diag = train.copy()
digit_diag["target_numeric"] = y.to_numpy()

income_int = (
    digit_diag["Annual_Income_USD"]
    .round()
    .astype("int64")
)

commute_tenths = (
    digit_diag["Daily_Commute_km"] * 10
).round().astype("int64")

# Income digit structure
digit_diag["Income_last_digit"] = income_int % 10
digit_diag["Income_last_2_digits"] = income_int % 100

digit_diag["Income_divisible_10"] = (income_int % 10 == 0).astype("int8")
digit_diag["Income_divisible_50"] = (income_int % 50 == 0).astype("int8")
digit_diag["Income_divisible_100"] = (income_int % 100 == 0).astype("int8")
digit_diag["Income_divisible_500"] = (income_int % 500 == 0).astype("int8")

# Commute decimal structure
digit_diag["Commute_tenths_digit"] = commute_tenths % 10

digit_diag["Commute_is_integer"] = (
    commute_tenths % 10 == 0
).astype("int8")

digit_diag["Commute_is_5"] = (
    digit_diag["Daily_Commute_km"] == 5.0
).astype("int8")

In [106]:
diagnostic_cols = [
    "Income_last_digit",
    "Income_divisible_10",
    "Income_divisible_50",
    "Income_divisible_100",
    "Income_divisible_500",
    "Commute_tenths_digit",
    "Commute_is_integer",
    "Commute_is_5",
]

for col in diagnostic_cols:
    print("\n" + "=" * 60)
    print(col)
    print("=" * 60)

    stats = (
        digit_diag
        .groupby(col)["target_numeric"]
        .agg(["size", "mean"])
        .rename(columns={
            "size": "rows",
            "mean": "purchase_rate"
        })
    )

    display(stats)


Income_last_digit


,rows,purchase_rate
Income_last_digit,,
0,133245,0.122001
1,65585,0.172951
2,52881,0.180897
3,59174,0.189560
4,57336,0.197869
5,63432,0.197913
6,57406,0.189092
7,59725,0.195948
8,59702,0.181334



Income_divisible_10


,rows,purchase_rate
Income_divisible_10,,
0,535420,0.187746
1,133245,0.122001



Income_divisible_50


...

,rows,purchase_rate
Commute_tenths_digit,,
0,195588,0.176417
1,49304,0.159987
2,48136,0.170912
3,53408,0.175273
4,59505,0.171179
5,52888,0.177866
6,48818,0.183662
7,54761,0.170541
8,51048,0.178459



Commute_is_integer


,rows,purchase_rate
Commute_is_integer,,
0,473077,0.173912
1,195588,0.176417



Commute_is_5


,rows,purchase_rate
Commute_is_5,,
0,524385,0.171954
1,144280,0.184426


In [107]:
income_int = train["Annual_Income_USD"].round().astype("int64")

income_rounding_class = np.select(
    [
        income_int % 500 == 0,
        income_int % 100 == 0,
        income_int % 50 == 0,
        income_int % 10 == 0,
    ],
    [
        "Divisible_500",
        "Divisible_100_not_500",
        "Divisible_50_not_100",
        "Divisible_10_not_50",
    ],
    default="Not_divisible_10"
)

rounding_diag = pd.DataFrame({
    "rounding_class": income_rounding_class,
    "target": y.to_numpy()
})

rounding_stats = (
    rounding_diag
    .groupby("rounding_class")["target"]
    .agg(["size", "mean"])
    .rename(columns={
        "size": "rows",
        "mean": "purchase_rate"
    })
    .sort_values("purchase_rate")
)

rounding_stats

,rows,purchase_rate
rounding_class,,
Divisible_500,62026,0.044739
Divisible_100_not_500,5434,0.158815
Divisible_50_not_100,8076,0.175582
Not_divisible_10,535420,0.187746
Divisible_10_not_50,57709,0.194077


In [108]:
income_last2_stats = (
    digit_diag
    .groupby("Income_last_2_digits")["target_numeric"]
    .agg(["size", "mean"])
    .rename(columns={
        "size": "rows",
        "mean": "purchase_rate"
    })
    .sort_values("rows", ascending=False)
)

display(income_last2_stats.head(30))

,rows,purchase_rate
Income_last_2_digits,,
0,67460,0.053928
78,9063,0.175108
80,8560,0.223131
77,8443,0.178254
60,8423,0.231628
36,8393,0.183963
45,8357,0.182841
31,8240,0.169903
50,8076,0.175582


In [109]:
large_last2 = income_last2_stats[
    income_last2_stats["rows"] >= 1000
]

print("Lowest purchase-rate endings:")
display(
    large_last2
    .sort_values("purchase_rate")
    .head(15)
)

print("\nHighest purchase-rate endings:")
display(
    large_last2
    .sort_values("purchase_rate", ascending=False)
    .head(15)
)

Lowest purchase-rate endings:


,rows,purchase_rate
Income_last_2_digits,,
0,67460,0.053928
71,4749,0.139819
69,6660,0.147748
38,5802,0.149259
92,4409,0.149467
1,6817,0.158134
49,6727,0.159061
99,6575,0.160608
51,7350,0.160816



Highest purchase-rate endings:


,rows,purchase_rate
Income_last_2_digits,,
85,4790,0.258664
60,8423,0.231628
7,2721,0.228960
87,6392,0.226377
5,3415,0.225769
80,8560,0.223131
16,5542,0.222483
4,3093,0.220821
19,7566,0.219667


In [110]:
income_last2_categories = [f"{i:02d}" for i in range(100)]

income_rounding_categories = [
    "Divisible_500",
    "Divisible_100_not_500",
    "Divisible_50_not_100",
    "Divisible_10_not_50",
    "Not_divisible_10",
]


def add_income_digit_features(df):
    df = df.copy()

    income_int = (
        df["Annual_Income_USD"]
        .round()
        .astype("int64")
    )

    # Last two digits as a true categorical feature
    last2 = (income_int % 100).map(lambda x: f"{x:02d}")

    df["Income_last_2_digits"] = pd.Categorical(
        last2,
        categories=income_last2_categories,
        ordered=False
    )

    # Mutually exclusive income-number-generation class
    rounding_class = np.select(
        [
            income_int % 500 == 0,
            income_int % 100 == 0,
            income_int % 50 == 0,
            income_int % 10 == 0,
        ],
        [
            "Divisible_500",
            "Divisible_100_not_500",
            "Divisible_50_not_100",
            "Divisible_10_not_50",
        ],
        default="Not_divisible_10"
    )

    df["Income_rounding_class"] = pd.Categorical(
        rounding_class,
        categories=income_rounding_categories,
        ordered=False
    )

    return df

In [111]:
train_v3 = add_income_digit_features(train)
test_v3 = add_income_digit_features(test)

print(
    train_v3[
        [
            "Annual_Income_USD",
            "Income_last_2_digits",
            "Income_rounding_class"
        ]
    ].head(10)
)

print("\nLast-2 unique:")
print(
    train_v3["Income_last_2_digits"].nunique(),
    test_v3["Income_last_2_digits"].nunique()
)

print("\nRounding classes:")
print(train_v3["Income_rounding_class"].value_counts())

   Annual_Income_USD Income_last_2_digits Income_rounding_class
0            92887.0                   87      Not_divisible_10
1            30000.0                   00         Divisible_500
2            94389.0                   89      Not_divisible_10
3            73580.0                   80   Divisible_10_not_50
4            57898.0                   98      Not_divisible_10
5            95938.0                   38      Not_divisible_10
6           109219.0                   19      Not_divisible_10
7            60796.0                   96      Not_divisible_10
8            67725.0                   25      Not_divisible_10
9           114827.0                   27      Not_divisible_10

Last-2 unique:
100 100

Rounding classes:
Income_rounding_class
Not_divisible_10         535420
Divisible_500             62026
Divisible_10_not_50       57709
Divisible_50_not_100       8076
Divisible_100_not_500      5434
Name: count, dtype: int64


In [113]:
# ============================================================
# V3 FEATURE CREATION
# Income digit / synthetic-generation artifacts
# ============================================================

income_last2_categories = [
    f"{i:02d}" for i in range(100)
]

income_rounding_categories = [
    "Divisible_500",
    "Divisible_100_not_500",
    "Divisible_50_not_100",
    "Divisible_10_not_50",
    "Not_divisible_10",
]


def add_income_digit_features(df):
    df = df.copy()

    income_int = (
        df["Annual_Income_USD"]
        .round()
        .astype("int64")
    )

    # --------------------------------------------------------
    # Last two digits
    # Example:
    # 84,785 -> "85"
    # 85,000 -> "00"
    # --------------------------------------------------------
    last2 = (
        income_int % 100
    ).map(
        lambda x: f"{x:02d}"
    )

    df["Income_last_2_digits"] = pd.Categorical(
        last2,
        categories=income_last2_categories,
        ordered=False
    )

    # --------------------------------------------------------
    # Mutually exclusive income rounding / generator class
    # --------------------------------------------------------
    rounding_class = np.select(
        [
            income_int % 500 == 0,
            income_int % 100 == 0,
            income_int % 50 == 0,
            income_int % 10 == 0,
        ],
        [
            "Divisible_500",
            "Divisible_100_not_500",
            "Divisible_50_not_100",
            "Divisible_10_not_50",
        ],
        default="Not_divisible_10"
    )

    df["Income_rounding_class"] = pd.Categorical(
        rounding_class,
        categories=income_rounding_categories,
        ordered=False
    )

    return df


# Create V3 train/test copies
train_v3 = add_income_digit_features(train)
test_v3 = add_income_digit_features(test)


# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
print("Train shape:", train_v3.shape)
print("Test shape:", test_v3.shape)

print("\nNew feature preview:")
display(
    train_v3[
        [
            "Annual_Income_USD",
            "Income_last_2_digits",
            "Income_rounding_class"
        ]
    ].head(10)
)

print("\nUnique last-two-digit groups:")
print(
    "Train:",
    train_v3["Income_last_2_digits"].nunique()
)
print(
    "Test:",
    test_v3["Income_last_2_digits"].nunique()
)

print("\nIncome rounding classes:")
print(
    train_v3["Income_rounding_class"]
    .value_counts()
)

Train shape: (668665, 17)
Test shape: (286571, 16)

New feature preview:


,Annual_Income_USD,Income_last_2_digits,Income_rounding_class
0,92887.0,87,Not_divisible_10
1,30000.0,00,Divisible_500
2,94389.0,89,Not_divisible_10
3,73580.0,80,Divisible_10_not_50
4,57898.0,98,Not_divisible_10
5,95938.0,38,Not_divisible_10
6,109219.0,19,Not_divisible_10
7,60796.0,96,Not_divisible_10
8,67725.0,25,Not_divisible_10
9,114827.0,27,Not_divisible_10



Unique last-two-digit groups:
Train: 100
Test: 100

Income rounding classes:
Income_rounding_class
Not_divisible_10         535420
Divisible_500             62026
Divisible_10_not_50       57709
Divisible_50_not_100       8076
Divisible_100_not_500      5434
Name: count, dtype: int64


In [114]:
# ============================================================
# V3 — INCOME DIGIT ARTIFACT MODEL
#
# V2:
#   income + commute artifact TE/frequency features
#
# V3 adds:
#   Income_last_2_digits
#   Income_rounding_class
#
# Everything else stays unchanged.
# ============================================================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier, early_stopping, log_evaluation


# ------------------------------------------------------------
# Outer CV
# ------------------------------------------------------------
outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# ------------------------------------------------------------
# Separate V3 containers
# IMPORTANT: these do NOT overwrite V1/V2 predictions
# ------------------------------------------------------------
oof_predictions_v3 = np.zeros(
    len(train_v3)
)

test_predictions_v3 = np.zeros(
    len(test_v3)
)

fold_scores_v3 = []
best_iterations_v3 = []


# ============================================================
# OUTER CV LOOP
# ============================================================

for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(train_v3, y),
    start=1
):

    print(
        f"\n{'=' * 60}"
    )
    print(
        f"OUTER FOLD {fold}"
    )
    print(
        f"{'=' * 60}"
    )

    # --------------------------------------------------------
    # Outer train / validation
    # --------------------------------------------------------
    outer_train = (
        train_v3
        .iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    outer_valid = (
        train_v3
        .iloc[valid_idx]
        .copy()
        .reset_index(drop=True)
    )

    outer_test = (
        test_v3
        .copy()
        .reset_index(drop=True)
    )

    y_outer_train = (
        y.iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    y_outer_valid = (
        y.iloc[valid_idx]
        .copy()
        .reset_index(drop=True)
    )


    # ========================================================
    # LEAK-FREE V2 SYNTHETIC ARTIFACT ENCODINGS
    #
    # Income:
    #   exact
    #   floor 50
    #   floor 250
    #   floor 500
    #
    # Commute:
    #   exact
    #   floor 1
    #   floor 5
    #   floor 10
    #
    # Each receives:
    #   nested OOF target encoding
    #   frequency encoding
    # ========================================================

    (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    ) = create_artifact_features_v2_three_way(
        outer_train,
        outer_valid,
        outer_test,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42 + fold
    )


    # ========================================================
    # ORDINARY ENGINEERED FEATURES
    # ========================================================

    train_fe_fold = (
        add_features(
            outer_train
        )
        .reset_index(drop=True)
    )

    valid_fe_fold = (
        add_features(
            outer_valid
        )
        .reset_index(drop=True)
    )

    test_fe_fold = (
        add_features(
            outer_test
        )
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Safety:
    # make absolutely sure V3 features survive add_features()
    # --------------------------------------------------------

    v3_special_features = [
        "Income_last_2_digits",
        "Income_rounding_class"
    ]

    for col in v3_special_features:

        train_fe_fold[col] = (
            outer_train[col]
            .to_numpy()
        )

        valid_fe_fold[col] = (
            outer_valid[col]
            .to_numpy()
        )

        test_fe_fold[col] = (
            outer_test[col]
            .to_numpy()
        )


    # ========================================================
    # BUILD MODEL MATRICES
    # ========================================================

    X_train_fold = (
        train_fe_fold
        .drop(
            columns=[
                "id",
                "Will_Buy_EV"
            ]
        )
        .reset_index(drop=True)
    )

    X_valid_fold = (
        valid_fe_fold
        .drop(
            columns=[
                "id",
                "Will_Buy_EV"
            ]
        )
        .reset_index(drop=True)
    )

    X_test_fold = (
        test_fe_fold
        .drop(
            columns=["id"]
        )
        .reset_index(drop=True)
    )


    # ========================================================
    # APPEND TARGET + FREQUENCY ENCODED ARTIFACTS
    # ========================================================

    X_train_fold = pd.concat(
        [
            X_train_fold,
            train_te,
            train_freq
        ],
        axis=1
    )

    X_valid_fold = pd.concat(
        [
            X_valid_fold,
            valid_te,
            valid_freq
        ],
        axis=1
    )

    X_test_fold = pd.concat(
        [
            X_test_fold,
            test_te,
            test_freq
        ],
        axis=1
    )


    # ========================================================
    # FIRST-FOLD SANITY CHECK
    # ========================================================

    if fold == 1:

        print(
            "\nV3 FEATURE CHECK"
        )

        print(
            "Income_last_2_digits present:",
            "Income_last_2_digits"
            in X_train_fold.columns
        )

        print(
            "Income_rounding_class present:",
            "Income_rounding_class"
            in X_train_fold.columns
        )

        print(
            "Train feature count:",
            X_train_fold.shape[1]
        )


    # ========================================================
    # ALIGN CATEGORICAL DTYPES
    # ========================================================

    categorical_cols = (
        X_train_fold
        .select_dtypes(
            include=[
                "object",
                "str",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    for col in categorical_cols:

        categories = (
            pd.concat(
                [
                    X_train_fold[
                        col
                    ].astype(str),

                    X_valid_fold[
                        col
                    ].astype(str),

                    X_test_fold[
                        col
                    ].astype(str)
                ],
                ignore_index=True
            )
            .unique()
            .tolist()
        )

        X_train_fold[col] = (
            pd.Categorical(
                X_train_fold[
                    col
                ].astype(str),
                categories=categories
            )
        )

        X_valid_fold[col] = (
            pd.Categorical(
                X_valid_fold[
                    col
                ].astype(str),
                categories=categories
            )
        )

        X_test_fold[col] = (
            pd.Categorical(
                X_test_fold[
                    col
                ].astype(str),
                categories=categories
            )
        )


    # ========================================================
    # LIGHTGBM
    # SAME MODEL AS V2
    # ========================================================

    model = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42 + fold,

        n_jobs=-1,
        verbosity=-1
    )


    # ========================================================
    # TRAIN
    # ========================================================

    model.fit(
        X_train_fold,
        y_outer_train,

        eval_X=X_valid_fold,
        eval_y=y_outer_valid,

        eval_metric="auc",

        callbacks=[
            early_stopping(
                150,
                first_metric_only=True
            ),

            log_evaluation(0)
        ]
    )


    # ========================================================
    # PREDICT
    # ========================================================

    valid_pred = (
        model.predict_proba(
            X_valid_fold
        )[:, 1]
    )

    test_pred = (
        model.predict_proba(
            X_test_fold
        )[:, 1]
    )


    # ========================================================
    # STORE OOF + TEST
    # ========================================================

    oof_predictions_v3[
        valid_idx
    ] = valid_pred

    test_predictions_v3 += (
        test_pred /
        outer_cv.n_splits
    )


    # ========================================================
    # FOLD SCORE
    # ========================================================

    fold_auc = roc_auc_score(
        y_outer_valid,
        valid_pred
    )

    fold_scores_v3.append(
        fold_auc
    )

    best_iterations_v3.append(
        model.best_iteration_
    )


    print(
        f"Fold {fold} AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )


OUTER FOLD 1

V3 FEATURE CHECK
Income_last_2_digits present: True
Income_rounding_class present: True
Train feature count: 42
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[507]	valid_0's auc: 0.944016	valid_0's binary_logloss: 0.222214
Evaluated only: auc
Fold 1 AUC: 0.944016
Best iteration: 507

OUTER FOLD 2
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[336]	valid_0's auc: 0.944779	valid_0's binary_logloss: 0.22073
Evaluated only: auc
Fold 2 AUC: 0.944779
Best iteration: 336

OUTER FOLD 3
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[436]	valid_0's auc: 0.945876	valid_0's binary_logloss: 0.21884
Evaluated only: auc
Fold 3 AUC: 0.945876
Best iteration: 436

OUTER FOLD 4
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[371]	valid_0's auc: 0.94542	valid_0's binary_logloss: 0.219584
Eva

In [115]:
# ============================================================
# V3 RESULTS
# ============================================================

v3_oof_auc = roc_auc_score(
    y,
    oof_predictions_v3
)

print(
    "\n" + "=" * 60
)

print(
    "V3 RESULTS — INCOME DIGIT ARTIFACTS"
)

print(
    "=" * 60
)

print(
    "Fold scores:"
)

for i, score in enumerate(
    fold_scores_v3,
    start=1
):
    print(
        f"Fold {i}: {score:.6f}"
    )

print()

print(
    f"Mean fold AUC: "
    f"{np.mean(fold_scores_v3):.6f}"
)

print(
    f"Fold std: "
    f"{np.std(fold_scores_v3):.6f}"
)

print(
    f"OOF AUC: "
    f"{v3_oof_auc:.9f}"
)

print(
    "\nBest iterations:"
)

print(
    best_iterations_v3
)


# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------

v1_auc_reference = 0.945510893
v2_auc_reference = 0.945599254
current_blend_reference = 0.945645

print(
    "\nCOMPARISON"
)

print(
    "-" * 60
)

print(
    f"V1 income artifacts:       "
    f"{v1_auc_reference:.9f}"
)

print(
    f"V2 income + commute:       "
    f"{v2_auc_reference:.9f}"
)

print(
    f"Current V1/V2 blend:       "
    f"~{current_blend_reference:.6f}"
)

print(
    f"V3 digit artifacts:        "
    f"{v3_oof_auc:.9f}"
)

print(
    f"\nV3 - V2: "
    f"{v3_oof_auc - v2_auc_reference:+.9f}"
)

print(
    f"V3 - V1: "
    f"{v3_oof_auc - v1_auc_reference:+.9f}"
)


# ============================================================
# SAVE V3 PREDICTIONS LOCALLY
# ============================================================

v3_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "v3_oof": oof_predictions_v3
})

v3_test_df = pd.DataFrame({
    "id": test["id"],
    "v3_prediction": test_predictions_v3
})


v3_oof_df.to_csv(
    "../outputs/predictions/income_digit_v3_oof.csv",
    index=False
)

v3_test_df.to_csv(
    "../outputs/predictions/income_digit_v3_test.csv",
    index=False
)

print(
    "\nV3 OOF and test predictions saved."
)


V3 RESULTS — INCOME DIGIT ARTIFACTS
Fold scores:
Fold 1: 0.944016
Fold 2: 0.944779
Fold 3: 0.945876
Fold 4: 0.945420
Fold 5: 0.945130

Mean fold AUC: 0.945044
Fold std: 0.000627
OOF AUC: 0.945037995

Best iterations:
[507, 336, 436, 371, 544]

COMPARISON
------------------------------------------------------------
V1 income artifacts:       0.945510893
V2 income + commute:       0.945599254
Current V1/V2 blend:       ~0.945645
V3 digit artifacts:        0.945037995

V3 - V2: -0.000561259
V3 - V1: -0.000472898

V3 OOF and test predictions saved.


In [116]:
v3_oof = pd.read_csv(
    "../outputs/predictions/income_digit_v3_oof.csv"
)

v3_pred = v3_oof["v3_oof"].to_numpy()

print("V3 AUC:", roc_auc_score(target, v3_pred))

print(
    "Pearson V1-V3:",
    np.corrcoef(v1_pred, v3_pred)[0, 1]
)

print(
    "Pearson V2-V3:",
    np.corrcoef(v2_pred, v3_pred)[0, 1]
)

print(
    "Spearman V1-V3:",
    spearmanr(v1_pred, v3_pred).statistic
)

print(
    "Spearman V2-V3:",
    spearmanr(v2_pred, v3_pred).statistic
)

V3 AUC: 0.94503799532578
Pearson V1-V3: 0.9961934145471558
Pearson V2-V3: 0.9971521387390192
Spearman V1-V3: 0.9933568874354252
Spearman V2-V3: 0.9939715941606955


In [118]:
v3_rank = pd.Series(v3_pred).rank(pct=True).to_numpy()

base_blend = (
    0.35 * v1_rank +
    0.65 * v2_rank
)

results_v3_blend = []

for v3_weight in np.arange(0.0, 0.31, 0.01):

    remaining = 1.0 - v3_weight

    blended = (
        remaining * base_blend +
        v3_weight * v3_rank
    )

    auc = roc_auc_score(
        target,
        blended
    )

    results_v3_blend.append({
        "V3_weight": v3_weight,
        "Base_weight": remaining,
        "AUC": auc
    })

results_v3_blend = pd.DataFrame(
    results_v3_blend
)

results_v3_blend.sort_values(
    "AUC",
    ascending=False
).head(20)

,V3_weight,Base_weight,AUC
9,0.09,0.91,0.945652
8,0.08,0.92,0.945652
10,0.10,0.90,0.945652
7,0.07,0.93,0.945652
11,0.11,0.89,0.945652
6,0.06,0.94,0.945651
12,0.12,0.88,0.945651
5,0.05,0.95,0.945651
13,0.13,0.87,0.945651
4,0.04,0.96,0.945650


In [119]:
# ============================================================
# V3A — ROUNDING CLASS ONLY
# ============================================================

income_rounding_categories = [
    "Divisible_500",
    "Divisible_100_not_500",
    "Divisible_50_not_100",
    "Divisible_10_not_50",
    "Not_divisible_10",
]


def add_income_rounding_feature(df):
    df = df.copy()

    income_int = (
        df["Annual_Income_USD"]
        .round()
        .astype("int64")
    )

    rounding_class = np.select(
        [
            income_int % 500 == 0,
            income_int % 100 == 0,
            income_int % 50 == 0,
            income_int % 10 == 0,
        ],
        [
            "Divisible_500",
            "Divisible_100_not_500",
            "Divisible_50_not_100",
            "Divisible_10_not_50",
        ],
        default="Not_divisible_10"
    )

    df["Income_rounding_class"] = pd.Categorical(
        rounding_class,
        categories=income_rounding_categories,
        ordered=False
    )

    return df


train_v3a = add_income_rounding_feature(train)
test_v3a = add_income_rounding_feature(test)

print(
    train_v3a["Income_rounding_class"]
    .value_counts()
)

print(
    "\nPresent:",
    "Income_rounding_class" in train_v3a.columns
)

print(
    "Last-two-digit accidentally present:",
    "Income_last_2_digits" in train_v3a.columns
)

Income_rounding_class
Not_divisible_10         535420
Divisible_500             62026
Divisible_10_not_50       57709
Divisible_50_not_100       8076
Divisible_100_not_500      5434
Name: count, dtype: int64

Present: True
Last-two-digit accidentally present: False


In [120]:
# ============================================================
# V3A — INCOME ROUNDING CLASS ONLY
#
# Baseline:
#   V2 income + commute artifact encodings
#
# New feature:
#   Income_rounding_class
#
# No Income_last_2_digits
# ============================================================

outer_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_predictions_v3a = np.zeros(len(train_v3a))
test_predictions_v3a = np.zeros(len(test_v3a))

fold_scores_v3a = []
best_iterations_v3a = []


for fold, (train_idx, valid_idx) in enumerate(
    outer_cv.split(train_v3a, y),
    start=1
):

    print(f"\n{'=' * 60}")
    print(f"OUTER FOLD {fold}")
    print(f"{'=' * 60}")

    # --------------------------------------------------------
    # Outer train / validation / test
    # --------------------------------------------------------
    outer_train = (
        train_v3a
        .iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    outer_valid = (
        train_v3a
        .iloc[valid_idx]
        .copy()
        .reset_index(drop=True)
    )

    outer_test = (
        test_v3a
        .copy()
        .reset_index(drop=True)
    )

    y_outer_train = (
        y.iloc[train_idx]
        .copy()
        .reset_index(drop=True)
    )

    y_outer_valid = (
        y.iloc[valid_idx]
        .copy()
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Same V2 leak-free artifact features
    # --------------------------------------------------------
    (
        train_te,
        valid_te,
        test_te,
        train_freq,
        valid_freq,
        test_freq
    ) = create_artifact_features_v2_three_way(
        outer_train,
        outer_valid,
        outer_test,
        y_outer_train,
        smoothing=50,
        n_splits=5,
        random_state=42 + fold
    )


    # --------------------------------------------------------
    # Ordinary engineered features
    # --------------------------------------------------------
    train_fe_fold = (
        add_features(outer_train)
        .reset_index(drop=True)
    )

    valid_fe_fold = (
        add_features(outer_valid)
        .reset_index(drop=True)
    )

    test_fe_fold = (
        add_features(outer_test)
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Explicitly preserve the V3a feature
    # --------------------------------------------------------
    train_fe_fold["Income_rounding_class"] = (
        outer_train["Income_rounding_class"]
        .to_numpy()
    )

    valid_fe_fold["Income_rounding_class"] = (
        outer_valid["Income_rounding_class"]
        .to_numpy()
    )

    test_fe_fold["Income_rounding_class"] = (
        outer_test["Income_rounding_class"]
        .to_numpy()
    )


    # --------------------------------------------------------
    # Build matrices
    # --------------------------------------------------------
    X_train_fold = (
        train_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_valid_fold = (
        valid_fe_fold
        .drop(columns=["id", "Will_Buy_EV"])
        .reset_index(drop=True)
    )

    X_test_fold = (
        test_fe_fold
        .drop(columns=["id"])
        .reset_index(drop=True)
    )


    # --------------------------------------------------------
    # Append V2 TE + frequency features
    # --------------------------------------------------------
    X_train_fold = pd.concat(
        [
            X_train_fold,
            train_te,
            train_freq
        ],
        axis=1
    )

    X_valid_fold = pd.concat(
        [
            X_valid_fold,
            valid_te,
            valid_freq
        ],
        axis=1
    )

    X_test_fold = pd.concat(
        [
            X_test_fold,
            test_te,
            test_freq
        ],
        axis=1
    )


    # --------------------------------------------------------
    # First-fold sanity check
    # --------------------------------------------------------
    if fold == 1:

        print("\nV3A FEATURE CHECK")

        print(
            "Income_rounding_class present:",
            "Income_rounding_class"
            in X_train_fold.columns
        )

        print(
            "Income_last_2_digits present:",
            "Income_last_2_digits"
            in X_train_fold.columns
        )

        print(
            "Train feature count:",
            X_train_fold.shape[1]
        )


    # --------------------------------------------------------
    # Align categorical dtypes
    # --------------------------------------------------------
    categorical_cols = (
        X_train_fold
        .select_dtypes(
            include=[
                "object",
                "str",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    for col in categorical_cols:

        categories = (
            pd.concat(
                [
                    X_train_fold[col].astype(str),
                    X_valid_fold[col].astype(str),
                    X_test_fold[col].astype(str)
                ],
                ignore_index=True
            )
            .unique()
            .tolist()
        )

        X_train_fold[col] = pd.Categorical(
            X_train_fold[col].astype(str),
            categories=categories
        )

        X_valid_fold[col] = pd.Categorical(
            X_valid_fold[col].astype(str),
            categories=categories
        )

        X_test_fold[col] = pd.Categorical(
            X_test_fold[col].astype(str),
            categories=categories
        )


    # --------------------------------------------------------
    # Same LightGBM as V2
    # --------------------------------------------------------
    model = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,

        subsample=0.9,
        colsample_bytree=0.9,

        reg_alpha=0.1,
        reg_lambda=1.0,

        random_state=42 + fold,
        n_jobs=-1,
        verbosity=-1
    )


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------
    model.fit(
        X_train_fold,
        y_outer_train,

        eval_X=X_valid_fold,
        eval_y=y_outer_valid,

        eval_metric="auc",

        callbacks=[
            early_stopping(
                150,
                first_metric_only=True
            ),
            log_evaluation(0)
        ]
    )


    # --------------------------------------------------------
    # Predict
    # --------------------------------------------------------
    valid_pred = model.predict_proba(
        X_valid_fold
    )[:, 1]

    test_pred = model.predict_proba(
        X_test_fold
    )[:, 1]


    # --------------------------------------------------------
    # Store predictions
    # --------------------------------------------------------
    oof_predictions_v3a[
        valid_idx
    ] = valid_pred

    test_predictions_v3a += (
        test_pred / outer_cv.n_splits
    )


    # --------------------------------------------------------
    # Score
    # --------------------------------------------------------
    fold_auc = roc_auc_score(
        y_outer_valid,
        valid_pred
    )

    fold_scores_v3a.append(
        fold_auc
    )

    best_iterations_v3a.append(
        model.best_iteration_
    )


    print(
        f"Fold {fold} AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Best iteration: "
        f"{model.best_iteration_}"
    )


OUTER FOLD 1

V3A FEATURE CHECK
Income_rounding_class present: True
Income_last_2_digits present: False
Train feature count: 41
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[1011]	valid_0's auc: 0.944655	valid_0's binary_logloss: 0.221004
Evaluated only: auc
Fold 1 AUC: 0.944655
Best iteration: 1011

OUTER FOLD 2
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[613]	valid_0's auc: 0.945351	valid_0's binary_logloss: 0.219646
Evaluated only: auc
Fold 2 AUC: 0.945351
Best iteration: 613

OUTER FOLD 3
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[901]	valid_0's auc: 0.946415	valid_0's binary_logloss: 0.21778
Evaluated only: auc
Fold 3 AUC: 0.946415
Best iteration: 901

OUTER FOLD 4
Training until validation scores don't improve for 150 rounds
Early stopping, best iteration is:
[667]	valid_0's auc: 0.945982	valid_0's binary_logloss: 0.2185

In [121]:
# ============================================================
# V3A RESULTS
# ============================================================

v3a_oof_auc = roc_auc_score(
    y,
    oof_predictions_v3a
)

print("\n" + "=" * 60)
print("V3A RESULTS — ROUNDING CLASS ONLY")
print("=" * 60)

for i, score in enumerate(
    fold_scores_v3a,
    start=1
):
    print(
        f"Fold {i}: {score:.6f}"
    )

print()

print(
    f"Mean fold AUC: "
    f"{np.mean(fold_scores_v3a):.6f}"
)

print(
    f"Fold std: "
    f"{np.std(fold_scores_v3a):.6f}"
)

print(
    f"OOF AUC: "
    f"{v3a_oof_auc:.9f}"
)

print(
    "\nBest iterations:"
)
print(best_iterations_v3a)


# ------------------------------------------------------------
# Comparison
# ------------------------------------------------------------

v1_auc_reference = 0.945510893
v2_auc_reference = 0.945599254
v3_auc_reference = 0.945037995
blend_reference = 0.945646

print("\nCOMPARISON")
print("-" * 60)

print(
    f"V1 income artifacts:          "
    f"{v1_auc_reference:.9f}"
)

print(
    f"V2 income + commute:          "
    f"{v2_auc_reference:.9f}"
)

print(
    f"V3 digits + rounding:         "
    f"{v3_auc_reference:.9f}"
)

print(
    f"V3a rounding only:            "
    f"{v3a_oof_auc:.9f}"
)

print(
    f"Current V1/V2 blend:          "
    f"~{blend_reference:.6f}"
)

print(
    f"\nV3a - V2: "
    f"{v3a_oof_auc - v2_auc_reference:+.9f}"
)

print(
    f"V3a - V3: "
    f"{v3a_oof_auc - v3_auc_reference:+.9f}"
)


# ------------------------------------------------------------
# Save predictions
# ------------------------------------------------------------

v3a_oof_df = pd.DataFrame({
    "id": train["id"],
    "target": y,
    "v3a_oof": oof_predictions_v3a
})

v3a_test_df = pd.DataFrame({
    "id": test["id"],
    "v3a_prediction": test_predictions_v3a
})


v3a_oof_df.to_csv(
    "../outputs/predictions/income_rounding_v3a_oof.csv",
    index=False
)

v3a_test_df.to_csv(
    "../outputs/predictions/income_rounding_v3a_test.csv",
    index=False
)

print(
    "\nV3a predictions saved."
)


V3A RESULTS — ROUNDING CLASS ONLY
Fold 1: 0.944655
Fold 2: 0.945351
Fold 3: 0.946415
Fold 4: 0.945982
Fold 5: 0.945619

Mean fold AUC: 0.945604
Fold std: 0.000594
OOF AUC: 0.945597347

Best iterations:
[1011, 613, 901, 667, 624]

COMPARISON
------------------------------------------------------------
V1 income artifacts:          0.945510893
V2 income + commute:          0.945599254
V3 digits + rounding:         0.945037995
V3a rounding only:            0.945597347
Current V1/V2 blend:          ~0.945646

V3a - V2: -0.000001907
V3a - V3: +0.000559352

V3a predictions saved.


## Artifact Investigation — Final Conclusions

### Best standalone artifact model
V2 — Income + Commute Artifacts

- OOF AUC: 0.945599254
- Income keys:
  - Exact
  - Floor 50
  - Floor 250
  - Floor 500
- Commute keys:
  - Exact
  - Floor 1
  - Floor 5
  - Floor 10
- Nested leak-free target encoding
- Frequency encoding

### Best public submission so far
Submission #006 — 35% V1 + 65% V2 rank blend

- OOF AUC: ~0.945646
- Public LB: 0.94589

### Digit artifact experiments

V3:
- Added Income_last_2_digits
- Added Income_rounding_class
- OOF AUC: 0.945037995
- Result: significant regression

V3a:
- Added Income_rounding_class only
- OOF AUC: 0.945597347
- Difference vs V2: -0.000001907

### Interpretation

Income rounding patterns showed strong target relationships in EDA, but the
existing income artifact encodings already capture nearly all useful rounding
information.

The 100-category last-two-digit feature caused substantial overfitting.

Therefore:
- Keep V2 as the final artifact feature family.
- Do not use V3 or V3a in the final model.
- Stop further handcrafted digit-feature exploration unless later evidence
  suggests otherwise.